# Лабораторная работа 1. Введение и воспроизводимость исследований

Вариант 16: Mushroom (OpenML). Задача: определить съедобность гриба по его признакам. Целевая колонка `class`: `e` означает съедобный гриб, `p` означает ядовитый.

Цель работы: построить воспроизводимый конвейер с `Pipeline`, `ColumnTransformer` и собственным трансформером. Все вычисления находятся в этом ноутбуке. Ячейки выполняются сверху вниз; рабочая папка ядра должна быть `notebooks`, пути указаны относительно неё.

## 1. Подготовка проекта

В проекте уже есть каталоги `data/raw`, `data/processed`, `notebooks`, `src`, `models`, `reports`, файл `.gitignore`, репозитории Git и DVC. Повторно инициализировать их не нужно. В `src` ничего не выносим.

Для создания окружения с зафиксированными зависимостями команды выполняются из корня проекта (существующее окружение можно просто активировать):

```bash
python -m venv venv
source venv/bin/activate
python -m pip install -r requirements.txt
jupyter lab notebooks/mushrooms_analyze.ipynb
```

При первоначальной настройке нового проекта используются `git init`, `dvc init`, а версии установленных библиотек фиксируются командой `python -m pip freeze > requirements.txt`. В `.gitignore` исключаются `venv/`, `__pycache__/` и `.ipynb_checkpoints/`.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path('..')
data_path = root / 'data/raw/mushroom.csv'

## 2. Загрузка и просмотр данных

Источник: [Mushroom, OpenML ID 24, версия 1](https://www.openml.org/d/24). Если CSV уже есть, используем его; иначе скачиваем и сохраняем в `data/raw`.

В имеющемся файле 8124 строки и 23 колонки: 22 категориальных признака и `class`. Колонку `bruises%3F` оставляем с исходным названием. Пропуски есть в `stalk-root` (2480 значений), поэтому искусственные пропуски не добавляем. Числовых признаков в исходных данных нет. Файл `minidataset.csv` относится к другим данным и здесь не используется.

In [ ]:
if not data_path.exists():
    dataset = fetch_openml(name='mushroom', version=1, as_frame=True)
    dataset.frame.to_csv(data_path, index=False)

df = pd.read_csv(data_path, na_values='?')
print('Размер:', df.shape)
df.head()

In [ ]:
df.info()
print('Пропуски:')
print(df.isna().sum())
print('Количество объектов каждого класса:')
print(df['class'].value_counts())

## 3. Разбиение данных

Выделяем 80% строк для обучения и 20% для проверки. Стратификация сохраняет соотношение классов, `random_state=42` фиксирует разбиение. Ядовитый гриб обозначим единицей, съедобный нулём. Предобработка будет обучаться только на обучающей выборке.

In [ ]:
X = df.drop(columns='class')
y = df['class'].map({'e': 0, 'p': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Обучающая выборка:', X_train.shape)
print('Тестовая выборка:', X_test.shape)

## 4. Собственный трансформер

По варианту нужно посчитать количество редких символов в `odor`. Считаем редкими значения с частотой меньше 5% в обучающей выборке. Каждое значение `odor` в этом датасете состоит из одного символа, поэтому количество равно 0 или 1. Трансформер добавляет числовой признак `odor_rare_count`, сохраняя исходный `odor`. Частоты определяются в `fit`, без использования тестовых данных.

In [ ]:
class RareOdorCounter(TransformerMixin, BaseEstimator):
    def __init__(self, threshold=0.05):
        self.threshold = threshold

    def fit(self, X, y=None):
        frequencies = X['odor'].value_counts(normalize=True)
        self.rare_symbols_ = frequencies[
            frequencies < self.threshold
        ].index
        return self

    def transform(self, X):
        X = X.copy()
        X['odor_rare_count'] = X['odor'].isin(
            self.rare_symbols_
        ).astype(int)
        return X

## 5. Конвейер обработки и модель

Для исходных категориальных признаков используем заполнение пропусков модой и `OneHotEncoder(handle_unknown='ignore')`. Для добавленного числового признака используем заполнение медианой и `StandardScaler`, как требуется в задании. В конце конвейера находится логистическая регрессия. Необязательный `GridSearchCV` не используем.

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, ['odor_rare_count']),
    ('categorical', categorical_pipeline, X.columns.tolist()),
])

model = Pipeline([
    ('rare_odor', RareOdorCounter()),
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

## 6. Обучение и оценка

Обучение начнётся только при выполнении следующей ячейки. Accuracy показывает долю правильных ответов, ROC-AUC оценивает разделение классов по вероятности ядовитости. Обе метрики считаются на отложенной тестовой выборке.

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print('Редкие символы odor:',
      model.named_steps['rare_odor'].rare_symbols_.tolist())

## 7. Сохранение модели и краткого отчёта

Сохраняем весь конвейер в `models/mushroom.joblib`. Отчёт с фактическими метриками записываем в `reports/README.md`, не изменяя существующий README проекта. При загрузке модели через `joblib.load` в новом ядре сначала нужно выполнить импорты и ячейку с определением `RareOdorCounter`.

In [ ]:
joblib.dump(model, root / 'models/mushroom.joblib')

report = f'''# Лабораторная работа 1. Mushroom

Вариант 16. Задача: классификация съедобности грибов.
Источник: https://www.openml.org/d/24 (версия 1).
Данные: {len(df)} строк, {X.shape[1]} категориальных признака.
Пропусков в stalk-root: {df['stalk-root'].isna().sum()}.

## Этапы
Данные разделены на train/test в отношении 80/20,
со стратификацией и random_state=42.
Собственный трансформер добавляет количество редких символов odor
(частота меньше 5% на train). Категории заполняются модой и кодируются
one-hot; числовой признак проходит заполнение медианой и StandardScaler.
Все этапы объединены в Pipeline с логистической регрессией.
Конвейер сохранён в models/mushroom.joblib.

## Метрики на тестовой выборке
- Accuracy: {accuracy:.4f}
- ROC-AUC: {roc_auc:.4f}

## Вывод
Модель правильно классифицировала {accuracy:.2%} тестовых объектов.
Pipeline объединяет создание признака, обработку пропусков,
кодирование и классификацию. Параметры предобработки определяются
только на train, тестовые данные не участвуют в обучении.
Результат относится к этому датасету и не гарантирует безопасное
определение съедобности реальных грибов.

После выполнения работы приложить скриншоты структуры проекта,
вывода dvc status, интерфейса Jupyter и полученных метрик.
'''
(root / 'reports/README.md').write_text(report, encoding='utf-8')
print(report)

## 8. Фиксация результатов в DVC и Git

После выполнения ноутбука команды запускаются вручную из корня проекта. DVC сохраняет версии данных и модели, Git хранит код и DVC-файлы.

```bash
dvc add data/raw/mushroom.csv models/mushroom.joblib
dvc status
git add notebooks/mushrooms_analyze.ipynb requirements.txt .gitignore .dvc .dvcignore
git add data/raw/mushroom.csv.dvc data/raw/.gitignore
git add models/mushroom.joblib.dvc models/.gitignore
git add -f reports/README.md
git commit -m "Complete Mushroom lab 1"
```

Для отчёта добавьте скриншоты структуры проекта, `dvc status`, интерфейса Jupyter и метрик. Значения метрик появятся только после запуска ячеек.